In [2]:
import numpy as np
import random
from collections import deque, defaultdict
import heapq

# ─── 地圖與參數 ─────────────────────────────────────────
WIDTH, HEIGHT = 21, 11
treasure_list = [(0,6),(3,16),(8,2),(10,2),(10,17)]
walls_coords  = set([
    (0,4),(0,5),(0,7),(0,9),(1,1),(1,2),(1,4),(1,9),(1,10),(1,14),(1,18),
    (2,1),(2,3),(2,5),(2,7),(2,8),(2,9),(2,11),(2,13),(2,15),(2,16),(2,17),(2,19),
    (3,2),(3,8),(3,11),(3,17),(4,1),(4,4),(4,6),(4,10),(4,13),(4,16),(4,17),(4,18),(4,20),
    (5,4),(5,5),(5,6),(5,8),(5,9),(5,14),(5,15),(6,1),(6,2),(6,3),(6,6),(6,8),(6,10),(6,15),(6,16),(6,17),(6,19),
    (7,4),(7,6),(7,8),(7,10),(7,11),(7,17),(7,19),(8,1),(8,4),(8,8),(8,10),(8,13),(8,15),(8,18),(8,19),
    (9,1),(9,2),(9,4),(9,6),(9,7),(9,17),(10,1),(10,4),(10,16),(10,19)
])
START, GOAL = (0,0), (20,10)

# Q-learning 超參數
MAX_EPISODES = 1000
MAX_STEPS    = 1000
EPSILON, EPS_DECAY, EPS_MIN = 0.9, 0.995, 0.01
ALPHA, GAMMA = 0.9, 0.995

# Prioritized Sweeping 參數
PLANNING_STEPS = 50       # 每個真實步之後規劃多少次
PRIORITY_THRESHOLD = 1e-4 # 只有 |δ| > 閾值 才入堆

ACTIONS = ['up','down','left','right']
NUM_ACTIONS = len(ACTIONS)
NUM_MASK    = 1 << len(treasure_list)
NUM_STATES  = WIDTH*HEIGHT * NUM_MASK

# Double Q 表
Q1 = np.zeros((NUM_STATES, NUM_ACTIONS))
Q2 = np.zeros((NUM_STATES, NUM_ACTIONS))

# Model & 前驅 集合
Model = {}  # (s,a) → (r, s′)
predecessors = defaultdict(list)  # s′ → [ (s,a), … ]

# 小頂堆，存 (−priority, s, a)
priority_queue = []

# 工具
def to_index(pos):    return pos[1]*WIDTH + pos[0]
def encode_state(pos,mask): return to_index(pos)*NUM_MASK + mask
def compute_target(pos,act):
    x,y = pos
    if act=='up':    y = max(0,y-1)
    elif act=='down':y = min(HEIGHT-1,y+1)
    elif act=='left':x = max(0,x-1)
    else:            x = min(WIDTH-1,x+1)
    return (x,y)

# 訓練主迴圈
best_steps, best_score, best_path = MAX_STEPS+1, -1, None
# 用于回退到“最多宝藏”时的那条轨迹
best_treasure_score, best_treasure_path = -1, None
epsilon = EPSILON

for ep in range(1, MAX_EPISODES+1):
    pos, mask = START, 0
    s = encode_state(pos,mask)
    path = [s]; score = 0

    for step in range(1, MAX_STEPS+1):
        # 1) ε-貪婪 + 障礙規避
        if random.random() < epsilon:
            a = random.randrange(NUM_ACTIONS)
        else:
            a = np.argmax(Q1[s] + Q2[s])
        new_pos = compute_target(pos, ACTIONS[a])
        if new_pos in walls_coords or new_pos == pos:
            # 撞牆就當無效，繼續下一個迴圈重新選
            continue

        # 2) 計算獎勵 & 更新 mask
        r = -1
        new_mask = mask
        if new_pos in treasure_list:
            i = treasure_list.index(new_pos)
            if not (mask & (1<<i)):
                new_mask |= (1<<i); r = +10; score +=1
        elif new_pos == GOAL:
            r = +58 if new_mask==(NUM_MASK-1) else -50

        s2 = encode_state(new_pos, new_mask)

        # 3) Double Q-Learning 本步更新
        if random.random() < 0.5:
            a2 = np.argmax(Q1[s2])
            td = r + GAMMA*Q2[s2,a2] - Q1[s,a]
            Q1[s,a] += ALPHA*td
        else:
            a2 = np.argmax(Q2[s2])
            td = r + GAMMA*Q1[s2,a2] - Q2[s,a]
            Q2[s,a] += ALPHA*td

        # 4) 存 Model 與 前驅
        Model[(s,a)] = (r, s2)
        predecessors[s2].append((s,a))

        # 5) 計算優先順序並推入堆
        priority = abs(td)
        if priority > PRIORITY_THRESHOLD:
            heapq.heappush(priority_queue, (-priority, s, a))

        # 6) Prioritized Sweeping 規劃更新
        for _ in range(PLANNING_STEPS):
            if not priority_queue: break
            neg_p, sp, ap = heapq.heappop(priority_queue)
            rp, sp2 = Model[(sp,ap)]
            # 用最新的 Q-表做一次更新
            # 這裡我們也用 Double Q 的策略選一邊去更新另一張
            if random.random()<0.5:
                anp = np.argmax(Q1[sp2])
                td2 = rp + GAMMA*Q2[sp2,anp] - Q1[sp,ap]
                Q1[sp,ap] += ALPHA*td2
            else:
                anp = np.argmax(Q2[sp2])
                td2 = rp + GAMMA*Q1[sp2,anp] - Q2[sp,ap]
                Q2[sp,ap] += ALPHA*td2

            # 對每個前驅再計算一次優先順序
            for spre, apre in predecessors[sp]:
                rpre, spre2 = Model[(spre,apre)]
                # 取這條 (spre,apre) 的“預測”td_error
                q_target = rpre + GAMMA * np.max((Q1 if random.random()<0.5 else Q2)[spre2])
                q_pred   = (Q1 if random.random()<0.5 else Q2)[spre,apre]
                p2 = abs(q_target - q_pred)
                if p2 > PRIORITY_THRESHOLD:
                    heapq.heappush(priority_queue, (-p2, spre, apre))

        # 7) 移動
        pos, mask, s = new_pos, new_mask, s2
        path.append(s)

        # 成功判斷
        if pos==GOAL and mask==(NUM_MASK-1):
            if step < best_steps:
                best_steps, best_score, best_path = step, score, path.copy()
            break

    # 每一轮实验结束后，更新「最多宝藏」那条轨迹
    if score > best_treasure_score:
        best_treasure_score = score
        best_treasure_path  = path.copy()

    # ε 衰減
    epsilon = max(EPS_MIN, epsilon*EPS_DECAY)
    # 印出訓練進度
    if ep % 50 == 0 or ep == 1 or ep == MAX_EPISODES:
        print(f"Ep{ep:4d} | ε={epsilon:.3f} | best_steps={best_steps} | best_score={best_score}")

# ─── 输出最終結果 ───────────────────────────────────────────
if best_path is not None:
    use_path, label, final_score = best_path, "（成功）", best_score
else:
    # 没有找到真正收集完+到终点的路径，就退而用「宝藏最多」的那条
    use_path, label, final_score = best_treasure_path, "（未成功，以宝藏数最多输出）", best_treasure_score

# 如果连 best_treasure_path 也没被更新（极端情况），就直接报错并退出
if use_path is None:
    print("⚠️ 连一条有效的轨迹都没有生成。")
    exit()    

# 重放 use_path，收集宝藏并记录最终位置
collected = []
pos, mask = START, 0
for state in use_path:
    idx = state // NUM_MASK
    x   = idx % WIDTH
    y   = idx // WIDTH

    if (x,y) in treasure_list and (x,y) not in collected:
        collected.append((x,y))
    pos = (x,y)

print(f"=== 最终结果 {label} ===")
print(f"步数：{len(use_path)-1}")
print(f"宝藏：{final_score}/{len(treasure_list)} -> {collected}")
print(f"最终停留：{pos} (应为 {GOAL})")

Ep   1 | ε=0.895 | best_steps=1001 | best_score=-1
Ep  50 | ε=0.700 | best_steps=1001 | best_score=-1
Ep 100 | ε=0.545 | best_steps=1001 | best_score=-1
Ep 150 | ε=0.424 | best_steps=1001 | best_score=-1
Ep 200 | ε=0.330 | best_steps=1001 | best_score=-1
Ep 250 | ε=0.257 | best_steps=1001 | best_score=-1
Ep 300 | ε=0.200 | best_steps=1001 | best_score=-1
Ep 350 | ε=0.156 | best_steps=1001 | best_score=-1
Ep 400 | ε=0.121 | best_steps=1001 | best_score=-1
Ep 450 | ε=0.094 | best_steps=1001 | best_score=-1
Ep 500 | ε=0.073 | best_steps=1001 | best_score=-1
Ep 550 | ε=0.057 | best_steps=1001 | best_score=-1
Ep 600 | ε=0.044 | best_steps=1001 | best_score=-1
Ep 650 | ε=0.035 | best_steps=1001 | best_score=-1
Ep 700 | ε=0.027 | best_steps=1001 | best_score=-1
Ep 750 | ε=0.021 | best_steps=1001 | best_score=-1
Ep 800 | ε=0.016 | best_steps=1001 | best_score=-1
Ep 850 | ε=0.013 | best_steps=1001 | best_score=-1
Ep 900 | ε=0.010 | best_steps=1001 | best_score=-1
Ep 950 | ε=0.010 | best_steps=1